> ## SOLUTION / ANSWER KEY &mdash; Lab 6.9
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-09-multi-tool.ipynb`](../lab-09-multi-tool.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.9 &mdash; Multi-Tool Orchestration (day in the life)

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Give the agent TWO tools: a search and a calculator
- Assemble them into one agent with create_agent
- Read a trace that chains them: search a fact, then compute on it

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; Connecting to real tools & APIs](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-09"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Agents earn their keep on **multi-step** tasks that **orchestrate several tools** (deck slide 16):
*"population of France, divided by 2?"* needs a **search** (a live fact) **then** a **calculator**. You
bind both tools to one agent; the agent chains them &mdash; each observation feeds the next step. The
graded cell checks the tools are wired and reads a fixed chained trace; the optional cell runs it live.

In [ ]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Search the web for a current fact."""
    facts = {"population of france": "68000000"}
    return facts.get(query.lower().strip(), "no result")

print("a search tool:", web_search.name, "->", web_search.invoke("population of france"))

## Your Turn
Give the agent **both** tools in `build_agent`, then complete `chain_is_grounded` &mdash; the check that the
**calculator step's argument was derived from the search step's result** (and that flags a BROKEN chain).

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

@tool
def web_search(query: str) -> str:
    """Search the web for a current fact."""
    facts = {"population of france": "68000000"}
    return facts.get(query.lower().strip(), "no result")

@tool
def calculator(expression: str) -> float:
    """Do exact arithmetic on an expression."""
    return safe_calc(expression)

def build_agent():
    model = ChatOllama(model="llama3.2:1b")
    tools = [web_search, calculator]
    return create_agent(model, tools)

# A GROUNDED trace: the calculator computed on the number search returned.
GROUNDED = [
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "population of france"}, "id": "a"}]),
    ToolMessage(content="68000000", tool_call_id="a"),
    AIMessage(content="", tool_calls=[{"name": "calculator", "args": {"expression": "68000000/2"}, "id": "b"}]),
    ToolMessage(content="34000000.0", tool_call_id="b"),
    AIMessage(content="34000000"),
]
# A BROKEN trace: the calculator argument was NOT derived from the search result (hallucinated).
BROKEN = [
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "population of france"}, "id": "a"}]),
    ToolMessage(content="68000000", tool_call_id="a"),
    AIMessage(content="", tool_calls=[{"name": "calculator", "args": {"expression": "42/2"}, "id": "b"}]),
    ToolMessage(content="21.0", tool_call_id="b"),
    AIMessage(content="21"),
]

def chained_arg(messages):
    """Pull (first tool observation, calculator expression) out of a search->calculator trace."""
    obs = next((m.content for m in messages if isinstance(m, ToolMessage)), None)
    expr = None
    for m in messages:
        for tc in (getattr(m, "tool_calls", None) or []):
            if tc["name"] == "calculator":
                expr = tc["args"]["expression"]
    return obs, expr

def chain_is_grounded(messages):
    obs, expr = chained_arg(messages)
    return bool(obs and expr and obs in expr)

try:
    agent = build_agent()
    print('agent nodes   :', set(agent.nodes) - {'__start__'})
    print('grounded chain:', chained_arg(GROUNDED), '->', chain_is_grounded(GROUNDED))
    print('broken chain  :', chained_arg(BROKEN), '->', chain_is_grounded(BROKEN))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("both tools are bound to the agent", lambda: type(build_agent()).__name__ == "CompiledStateGraph" and "tools" in set(build_agent().nodes))
expect_true("a GROUNDED chain is accepted: calc arg derived from the search result", lambda: chain_is_grounded(GROUNDED) is True)
expect_true("a BROKEN chain is detected: calc arg NOT grounded in the search result", lambda: chain_is_grounded(BROKEN) is False)
expect_true("the data dependency is read straight from the trace", lambda: chained_arg(GROUNDED) == ("68000000", "68000000/2"))
expect_true("the search tool returns the population the calculator consumes", lambda: web_search.invoke("population of france") == "68000000")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run the two-tool agent live and check whether it actually grounded the compute step in the search result. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    if ollama_up():
        agent = build_agent()
        result = agent.invoke({"messages": [{"role": "user",
                 "content": "Use web_search for the population of france, then use the calculator to halve it."}]},
                 config={"recursion_limit": 8})
        print("data dependency:", chained_arg(result["messages"]))
        print("grounded?      :", chain_is_grounded(result["messages"]))
        print("final          :", result["messages"][-1].content)
    else:
        print("No Ollama reachable -- skipping the live run. The wiring + grounding check above already work.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
Chaining search + compute in one run is exactly where agents beat a single call. Same create_agent, two tools -- the 'day in the life' trace, for real.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>